# Keyword Search Explorer

Interactive notebook to filter papers from [LeMat-Synth-Papers](https://huggingface.co/datasets/LeMaterial/LeMat-Synth-Papers).

**Pipeline:**
1. Load the three raw splits (`arxiv`, `omg24`, `chemrxiv`)
2. Pick a split
3. Explore & filter by **categories**
4. Filter by **keywords** (Option 1: direct string matching, Option 2: LLM-based)
5. Browse & export results

In [1]:
from datasets import load_dataset
import pandas as pd

pd.set_option("display.max_colwidth", 120)

# Load the three raw splits from HuggingFace
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",
    split=None,
    token=True,
)

print("Available splits:")
for name, ds in dataset.items():
    print(f"  {name}: {len(ds)} papers")
print(f"\nColumns: {dataset[list(dataset.keys())[0]].column_names}")

Available splits:
  arxiv: 62952 papers
  omg24: 16026 papers
  chemrxiv: 1962 papers
  thermocatalysis_keywords_only: 8378 papers
  thermocatalysis_keywords_and_LLM: 1333 papers

Columns: ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis']


---
## Step 1 — Choose a split

In [2]:
# Pick a split: "arxiv", "omg24", or "chemrxiv"
SPLIT = "arxiv"

ds = dataset[SPLIT]
print(f"Selected split '{SPLIT}': {len(ds)} papers")

Selected split 'arxiv': 62952 papers


---
## Step 2 — Explore categories

The `categories` column is a string. Individual categories are separated by `,` or `;`.

- **arxiv**: codes like `cond-mat.mtrl-sci`, `physics.chem-ph`
- **chemrxiv**: names like `Solid State Chemistry`, `Surface`
- **omg24**: Semantic Scholar field-of-study labels

In [3]:
def parse_categories(cat_string):
    """Split a categories string by comma and semicolon into a list."""
    if cat_string is None:
        return []
    parts = []
    for part in cat_string.split(","):
        for sub_part in part.split(";"):
            cleaned = sub_part.strip()
            if cleaned:
                parts.append(cleaned)
    return parts


# Flatten all categories across the split
all_cats = []
for cat_string in ds["categories"]:
    all_cats.extend(parse_categories(cat_string))

cat_counts = pd.Series(all_cats).value_counts()
print(f"'{SPLIT}' has {len(cat_counts)} unique categories across {len(ds)} papers\n")
cat_counts

'arxiv' has 915 unique categories across 62952 papers



Nanomaterials                        12331
Superconductors                      12148
Magnetic                             11937
Semiconductors                       10685
Ceramics                              6661
                                     ...  
Heterostructures                         1
Magnetic/Oxides                          1
Nonlinear Optical Materials              1
Heusler alloys (Magnetic)                1
Composites/Nanomaterials/Magnetic        1
Name: count, Length: 915, dtype: int64

## Step 3 — Filter by categories

Keep only papers whose `categories` string contains at least one match.  
Uses substring matching (case-insensitive).  
Set `CATEGORY_FILTER = []` to skip and keep all papers.

In [5]:
# Categories to keep (substring match, case-insensitive)
# Examples:
#   arxiv:    ["cond-mat"]
#   chemrxiv: ["Solid State Chemistry", "Surface"]
CATEGORY_FILTER = ["Superconductors"]  # empty = keep all papers


def filter_by_category(ds, category_filter):
    """Keep papers whose categories string contains any of the filter terms."""
    if not category_filter:
        return ds
    cat_lower = [c.lower() for c in category_filter]

    def has_category(example):
        cats = example["categories"]
        if cats is None:
            return False
        cats_lower = cats.lower()
        return any(cf in cats_lower for cf in cat_lower)

    return ds.filter(has_category)


ds_cat = filter_by_category(ds, CATEGORY_FILTER)

if CATEGORY_FILTER:
    print(f"Category filter {CATEGORY_FILTER}: {len(ds_cat)} / {len(ds)} papers")
else:
    print(f"No category filter applied: {len(ds_cat)} papers")

Filter:   0%|          | 0/62952 [00:00<?, ? examples/s]

Category filter ['Superconductors']: 12259 / 62952 papers


---
## Step 4 — Keyword filter

### Option 1: Direct keyword matching (default)

Papers matching **any** include keyword (and **none** of the exclude keywords) are kept.

In [6]:
# Column to search in
TEXT_COLUMN = "abstract"

# Include keywords — papers matching ANY of these are kept
INCLUDE_KEYWORDS = [
    "superconductor",
    "resistivity"
]

# Exclude keywords — papers matching ANY of these are removed ([] to skip)
EXCLUDE_KEYWORDS = ["semiconductor"]

# How many papers to display
NUM_DISPLAY = 20

In [7]:
def keyword_filter(ds, text_column, include_kws, exclude_kws=None):
    """Filter HuggingFace Dataset by include/exclude keywords."""
    def matches_include(example):
        if example[text_column] is None:
            return False
        text_lower = example[text_column].lower()
        return any(kw.lower() in text_lower for kw in include_kws)

    filtered = ds.filter(matches_include)

    if exclude_kws:
        def matches_exclude(example):
            if example[text_column] is None:
                return False
            text_lower = example[text_column].lower()
            return any(ek.lower() in text_lower for ek in exclude_kws)
        filtered = filtered.filter(lambda x: not matches_exclude(x))

    return filtered


ds_filtered = keyword_filter(ds_cat, TEXT_COLUMN, INCLUDE_KEYWORDS, EXCLUDE_KEYWORDS)
print(f"Keyword filter: {len(ds_filtered)} / {len(ds_cat)} papers")

Filter:   0%|          | 0/12259 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6486 [00:00<?, ? examples/s]

Keyword filter: 6384 / 12259 papers


### Option 2: LLM-based keyword filter (commented out)

Instead of exact string matching, use an LLM to decide whether a paper is relevant.  
Requires a running **vLLM** endpoint. Uncomment the cell below to use it.

**Prerequisites:**
- A vLLM server running locally (default: `http://localhost:8000/v1`)
- `pip install openai transformers`

In [8]:
# # --- Option 2: LLM-based filtering ---
# # Uncomment this entire cell to use LLM filtering instead of direct keywords.
# # Make sure to skip the Option 1 cells above.
#
# import openai
# from tqdm import tqdm
# from transformers import AutoTokenizer
#
# VLLM_ENDPOINT = "http://localhost:8000/v1"
# MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
# MAX_MODEL_LEN = 50000
#
# # Edit this prompt to match your use case
# LLM_PROMPT = """You are provided with a scientific materials paper.
# Read the paper carefully and determine if it is relevant to the topic
# of heterogeneous catalysis with temperature-dependent performance data.
#
# Answer with only yes or no.
# If you are not sure, answer with no.
#
# Paper: {paper_text}
# Question: Is this paper relevant?
# Answer:"""
#
# client = openai.OpenAI(base_url=VLLM_ENDPOINT, api_key="not-needed")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
#
# def ask_llm(text):
#     """Send paper text to LLM and return True if relevant."""
#     tokens = tokenizer.encode(text)
#     if len(tokens) > MAX_MODEL_LEN:
#         text = tokenizer.decode(tokens[:MAX_MODEL_LEN - 150])
#     message = LLM_PROMPT.format(paper_text=text)
#     try:
#         response = client.chat.completions.create(
#             model=MODEL_NAME,
#             messages=[{"role": "user", "content": message}],
#             temperature=0,
#             max_tokens=100,
#         )
#         answer = response.choices[0].message.content.strip().lower()
#         return "yes" in answer
#     except Exception as e:
#         print(f"LLM call failed: {e}")
#         return False
#
#
# # Run LLM filter on the category-filtered dataset (ds_cat)
# llm_results = []
# for paper in tqdm(ds_cat, desc="LLM filtering"):
#     text = paper.get("text_paper") or paper.get("abstract") or ""
#     llm_results.append(ask_llm(text))
#
# ds_cat_with_labels = ds_cat.add_column("llm_relevant", llm_results)
# ds_filtered = ds_cat_with_labels.filter(lambda x: x["llm_relevant"])
# print(f"LLM filter: {len(ds_filtered)} / {len(ds_cat)} papers")

---
## Step 5 — Results

### Per-keyword match counts

In [9]:
# Per-keyword counts on the filtered dataset
def count_keyword_matches(ds, text_column, keywords):
    counts = {}
    for kw in keywords:
        def has_keyword(example, keyword=kw):
            if example[text_column] is None:
                return False
            return keyword.lower() in example[text_column].lower()
        count = len(ds.filter(has_keyword))
        counts[kw] = count
    return counts


keyword_counts = count_keyword_matches(ds_filtered, TEXT_COLUMN, INCLUDE_KEYWORDS)
print("Per-keyword match counts:")
for kw, count in sorted(keyword_counts.items(), key=lambda x: -x[1]):
    print(f"  {kw}: {count}")
print(f"\nTotal papers: {len(ds_filtered)}")

Filter:   0%|          | 0/6384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6384 [00:00<?, ? examples/s]

Per-keyword match counts:
  superconductor: 5305
  resistivity: 1900

Total papers: 6384


### Browse results

In [10]:
DISPLAY_COLUMNS = ["id", "title", "categories", "abstract", "pdf_url"]

cols = [c for c in DISPLAY_COLUMNS if c in ds_filtered.column_names]
ds_filtered.to_pandas()[cols].head(NUM_DISPLAY)

,id,title,categories,abstract,pdf_url
0,0704.0352,Investigation of relaxation phenomena in high-temperature\n superconductors HoBa2Cu3O7-d at the action of pulsed ma...,Superconductors,It is used the mechanical method of Abrikosov vortex stimulated dynamics\ninvestigation in superconductors. With i...,http://arxiv.org/pdf/0704.0352v1
1,0704.0529,Scanning Tunneling Spectroscopy in the Superconducting State and Vortex\n Cores of the beta-pyrochlore KOs2O6,Superconductors,We performed the first scanning tunneling spectroscopy measurements on the\npyrochlore superconductor KOs2O6 (Tc =...,http://arxiv.org/pdf/0704.0529v1
2,0704.0694,Current - voltage characteristics of break junctions of high-$T_c$\n superconductors,Superconductors,The current-voltage ($I$-$V$) characteristics of break junctions of\npolycrystalline La$_{1.85}$Sr$_{0.15}$CuO$_4$...,http://arxiv.org/pdf/0704.0694v1
3,0704.0765,Evidence of Spatially Inhomogeous Pairing on the Insulating Side of a\n Disorder-Tuned Superconductor-Insulator Tra...,Superconductors,Measurements of transport properties of amorphous insulating indium oxide\nthin films have been interpreted as evi...,http://arxiv.org/pdf/0704.0765v2
4,0704.1528,Extremely strong-coupling superconductivity and anomalous lattice\n properties in the beta-pyrochlore oxide KOs2O6,Superconductors,Superconducting and normal-state properties of the beta-pyrochlore oxide\nKOs2O6 are studied by means of thermodyn...,http://arxiv.org/pdf/0704.1528v1
5,0704.1872,Evidence for nonmonotonic magnetic field penetration in a type-I\n superconductor,Superconductors,Polarized neutron reflectometry (PNR) provides evidence that nonlocal\nelectrodynamics governs the magnetic field ...,http://arxiv.org/pdf/0704.1872v2
6,0704.1970,Thermoelectric response near a quantum critical point: the case of\n CeCoIn5,Superconductors,We present a study of thermoelectric coefficients in CeCoIn_5 down to 0.1 K\nand up to 16 T in order to probe the ...,http://arxiv.org/pdf/0704.1970v2
7,0704.2336,Competition between unconventional superconductivity and incommensurate\n antiferromagnetic order in CeRh1-xCoxIn5,Superconductors,Elastic neutron diffraction measurements were performed on the quasi-two\ndimensional heavy fermion system CeRh1-x...,http://arxiv.org/pdf/0704.2336v2
8,0704.3364,Upper limit on spontaneous supercurrents in Sr$_2$RuO$_4$,Superconductors,It is widely believed that the perovskite Sr$_2$RuO$_4$ is an unconventional\nsuperconductor with broken time reve...,http://arxiv.org/pdf/0704.3364v1
9,0704.3526,MgB2 single crystals substituted with Li and with Li-C: Structural and\n superconducting properties,Superconductors,The effect of Li substitution for Mg and of Li-C co-substitution on the\nsuperconducting properties and crystal st...,http://arxiv.org/pdf/0704.3526v1


### Export (optional)

In [ ]:
# df = ds_filtered.to_pandas()
# df.to_csv("filtered_papers.csv", index=False)
# df.to_pickle("filtered_papers.pkl")